# 🔍 Dual-Stream CNN Training - CIFAKE Dataset
## Phát hiện hình ảnh giả mạo (Deepfake Detection)

**Kiến trúc:**
- Spatial Stream (RGB)
- Frequency Stream (FFT)
- Fusion Layer

⚡ **Chạy với GPU miễn phí trên Google Colab!**

## 1️⃣ Kiểm tra GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ Tải CIFAKE Dataset từ Kaggle

In [ ]:
# Cài đặt Kaggle
!pip install kaggle -q

# Upload kaggle.json
from google.colab import files
print("Vui lòng upload file kaggle.json")
print("(Lấy từ: https://www.kaggle.com/settings -> API -> Create New Token)")
uploaded = files.upload()

In [ ]:
# Cấu hình Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Tải CIFAKE dataset
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d cifake_data
!ls cifake_data

## 3️⃣ Chuẩn bị Dataset

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

def prepare_dataset():
    base_dir = 'dataset'
    cifake_dir = 'cifake_data'
    
    # Tạo thư mục
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{base_dir}/{split}/0_real', exist_ok=True)
        os.makedirs(f'{base_dir}/{split}/1_fake', exist_ok=True)
    
    # Copy REAL từ train
    real_src = f'{cifake_dir}/train/REAL'
    files = os.listdir(real_src)
    random.shuffle(files)
    split_idx = int(len(files) * 0.9)
    
    print(f"REAL: {len(files)} files (train: {split_idx}, val: {len(files)-split_idx})")
    for i, f in enumerate(tqdm(files, desc='REAL')):
        dst_split = 'train' if i < split_idx else 'val'
        shutil.copy2(f'{real_src}/{f}', f'{base_dir}/{dst_split}/0_real/{f}')
    
    # Copy FAKE từ train
    fake_src = f'{cifake_dir}/train/FAKE'
    files = os.listdir(fake_src)
    random.shuffle(files)
    split_idx = int(len(files) * 0.9)
    
    print(f"FAKE: {len(files)} files (train: {split_idx}, val: {len(files)-split_idx})")
    for i, f in enumerate(tqdm(files, desc='FAKE')):
        dst_split = 'train' if i < split_idx else 'val'
        shutil.copy2(f'{fake_src}/{f}', f'{base_dir}/{dst_split}/1_fake/{f}')
    
    # Copy test
    for cls in ['REAL', 'FAKE']:
        dst_cls = '0_real' if cls == 'REAL' else '1_fake'
        src = f'{cifake_dir}/test/{cls}'
        for f in tqdm(os.listdir(src), desc=f'Test {cls}'):
            shutil.copy2(f'{src}/{f}', f'{base_dir}/test/{dst_cls}/{f}')
    
    # Thống kê
    print("\n" + "="*50)
    for split in ['train', 'val', 'test']:
        real = len(os.listdir(f'{base_dir}/{split}/0_real'))
        fake = len(os.listdir(f'{base_dir}/{split}/1_fake'))
        print(f"{split}: {real} real + {fake} fake = {real+fake}")

prepare_dataset()

## 4️⃣ Định nghĩa Dual-Stream CNN Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class SpatialStream(nn.Module):
    """Nhánh Spatial: Xử lý ảnh RGB"""
    def __init__(self):
        super(SpatialStream, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.out_features = 512
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.avgpool(x)
        return torch.flatten(x, 1)


class FrequencyStream(nn.Module):
    """Nhánh Frequency: Xử lý phổ FFT"""
    def __init__(self):
        super(FrequencyStream, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.out_features = 512
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.avgpool(x)
        return torch.flatten(x, 1)


class DualStreamCNN(nn.Module):
    """Dual-Stream CNN: Kết hợp Spatial và Frequency"""
    def __init__(self, num_classes=1, dropout=0.5):
        super(DualStreamCNN, self).__init__()
        
        self.spatial_stream = SpatialStream()
        self.frequency_stream = FrequencyStream()
        
        combined_features = 512 + 512
        
        self.fusion = nn.Sequential(
            nn.Linear(combined_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, rgb_input, fft_input):
        spatial_features = self.spatial_stream(rgb_input)
        freq_features = self.frequency_stream(fft_input)
        combined = torch.cat([spatial_features, freq_features], dim=1)
        output = self.fusion(combined)
        return output


def compute_fft_spectrum(image_tensor):
    """Tính phổ FFT từ ảnh RGB"""
    if image_tensor.dim() == 4:
        gray = 0.299 * image_tensor[:, 0] + 0.587 * image_tensor[:, 1] + 0.114 * image_tensor[:, 2]
    else:
        gray = 0.299 * image_tensor[0] + 0.587 * image_tensor[1] + 0.114 * image_tensor[2]
    
    fft = torch.fft.fft2(gray)
    fft_shift = torch.fft.fftshift(fft)
    magnitude = torch.abs(fft_shift)
    magnitude = torch.log1p(magnitude)
    
    if magnitude.dim() == 3:
        for i in range(magnitude.shape[0]):
            magnitude[i] = (magnitude[i] - magnitude[i].min()) / (magnitude[i].max() - magnitude[i].min() + 1e-8)
    else:
        magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)
    
    if magnitude.dim() == 2:
        magnitude = magnitude.unsqueeze(0)
    else:
        magnitude = magnitude.unsqueeze(1)
    
    return magnitude


# Test model
model = DualStreamCNN()
rgb = torch.randn(2, 3, 32, 32)
fft = compute_fft_spectrum(rgb)
out = model(rgb, fft)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input RGB: {rgb.shape}, FFT: {fft.shape}")
print(f"Output: {out.shape}")

## 5️⃣ Dataset và DataLoader

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


def compute_fft_from_pil(image, size=32):
    """Tính FFT từ PIL Image"""
    image = image.resize((size, size), Image.LANCZOS)
    gray = image.convert('L')
    gray_array = np.array(gray, dtype=np.float32) / 255.0
    
    fft = np.fft.fft2(gray_array)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.abs(fft_shift)
    magnitude = np.log1p(magnitude)
    magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)
    
    return torch.from_numpy(magnitude).float().unsqueeze(0)


class DualStreamDataset(Dataset):
    def __init__(self, root_dir, transform=None, image_size=32):
        self.root_dir = root_dir
        self.image_size = image_size
        self.samples = []
        
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform
        
        # Load samples
        for label, cls_name in [(0, '0_real'), (1, '1_fake')]:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.exists(cls_dir):
                for fname in os.listdir(cls_dir):
                    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.samples.append((os.path.join(cls_dir, fname), label))
        
        print(f"Loaded {len(self.samples)} samples from {root_dir}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        fft_spectrum = compute_fft_from_pil(image, self.image_size)
        rgb_tensor = self.transform(image)
        
        return rgb_tensor, fft_spectrum, torch.tensor(label, dtype=torch.float32)


# Test dataset
train_dataset = DualStreamDataset('dataset/train', image_size=32)
val_dataset = DualStreamDataset('dataset/val', image_size=32)

## 6️⃣ Training

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
import torch.optim as optim

# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 30
LR = 0.0001
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Model
model = DualStreamCNN(num_classes=1, dropout=0.5).to(DEVICE)

# Loss and Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Training loop
best_auc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for rgb, fft, labels in pbar:
        rgb, fft, labels = rgb.to(DEVICE), fft.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(rgb, fft)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(outputs).squeeze().detach().cpu().numpy()
        train_preds.extend(preds.flatten().tolist() if preds.ndim > 0 else [preds.item()])
        train_labels.extend(labels.cpu().numpy().flatten().tolist())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(train_loader)
    train_acc = accuracy_score(train_labels, (np.array(train_preds) > 0.5).astype(int))
    
    # Validation
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for rgb, fft, labels in tqdm(val_loader, desc='Validation'):
            rgb, fft, labels = rgb.to(DEVICE), fft.to(DEVICE), labels.to(DEVICE)
            outputs = model(rgb, fft)
            loss = criterion(outputs.squeeze(), labels)
            
            val_loss += loss.item()
            preds = torch.sigmoid(outputs).squeeze().cpu().numpy()
            val_preds.extend(preds.flatten().tolist() if preds.ndim > 0 else [preds.item()])
            val_labels.extend(labels.cpu().numpy().flatten().tolist())
    
    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_labels, (np.array(val_preds) > 0.5).astype(int))
    val_auc = roc_auc_score(val_labels, val_preds)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}")
    
    # Save best model
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_auc': best_auc,
        }, 'best_dual_stream_model.pth')
        print(f"  ✓ Saved best model (AUC: {best_auc:.4f})")

print(f"\n" + "="*50)
print(f"Training Complete!")
print(f"Best Validation AUC: {best_auc:.4f}")
print("="*50)

## 7️⃣ Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True)

# AUC
axes[2].plot(history['val_auc'], label='Val AUC', color='green')
axes[2].set_title('Validation AUC')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 8️⃣ Download Model

In [ ]:
from google.colab import files

# Download model
files.download('best_dual_stream_model.pth')
files.download('training_history.png')

## 9️⃣ Test trên Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
checkpoint = torch.load('best_dual_stream_model.pth')
model.load_state_dict(checkpoint['model'])
model.eval()

# Test dataset
test_dataset = DualStreamDataset('dataset/test', image_size=32)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Predict
test_preds, test_labels = [], []
with torch.no_grad():
    for rgb, fft, labels in tqdm(test_loader, desc='Testing'):
        rgb, fft = rgb.to(DEVICE), fft.to(DEVICE)
        outputs = model(rgb, fft)
        preds = torch.sigmoid(outputs).squeeze().cpu().numpy()
        test_preds.extend(preds.flatten().tolist() if preds.ndim > 0 else [preds.item()])
        test_labels.extend(labels.numpy().flatten().tolist())

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)
test_pred_labels = (test_preds > 0.5).astype(int)

# Metrics
print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
print(f"Accuracy: {accuracy_score(test_labels, test_pred_labels):.4f}")
print(f"AUC: {roc_auc_score(test_labels, test_preds):.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_pred_labels, target_names=['Real', 'Fake']))

# Confusion Matrix
cm = confusion_matrix(test_labels, test_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()